# Text-to-SQL Fine-Tuning Workflow

This notebook is the polished portfolio walkthrough for the project. It summarizes the completed fine-tuning workflow using `Qwen/Qwen2.5-Coder-1.5B-Instruct`, a custom 2,503-example text-to-SQL dataset, and Spider-style evaluation.

The original exploratory notebooks are archived in `experiments/`. The trained checkpoint is intentionally ignored because the public repository is meant to demonstrate the workflow and methodology, not distribute model weights.

## 1. Setup

The workflow uses Hugging Face `transformers` and `datasets`. Training requires a CUDA GPU. The cells below are kept as reproducible workflow code, but this cleanup did not rerun training or evaluation.

In [ ]:
import os
from pathlib import Path

import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    default_data_collator,
)

BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
DATA_FILE = "spider_style_dataset.jsonl"
OUTPUT_DIR = "qwen_spider_full_sft_v3"
MAX_SEQ_LEN = 512

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

## 2. Dataset Loading

The dataset is stored as chat-format JSONL. Each row has a system prompt, a user message containing schema plus question, and an assistant message containing the target SQL query.

In [ ]:
raw_ds = load_dataset("json", data_files=DATA_FILE, split="train")
len(raw_ds), raw_ds[0]

**Recorded dataset summary:**

- `2,503` examples
- `397` schema/table combinations observed during artifact checks
- All committed rows use the expected `system -> user -> assistant` message pattern
- Assistant answers are SQL statements beginning with `SELECT` or `WITH`

## 3. Model and Tokenizer

The model is loaded in half precision/bfloat16 when available. Checkpoints are written to `qwen_spider_full_sft_v3/`, which is intentionally ignored by Git.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False
model.gradient_checkpointing_enable()

## 4. Prompt-Masked Tokenization

The corrected SFT path masks prompt tokens with `-100`, so the loss is only computed on the assistant SQL answer. This replaced the early prototype in `experiments/fine_tune_initial_full_text_training.ipynb`, which trained over the full chat text and produced unstable loss.

In [ ]:
def build_prompt_and_answer(example):
    messages = example["messages"]
    assert messages[-1]["role"] == "assistant"

    answer = messages[-1]["content"]
    prompt_messages = messages[:-1]
    prompt = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    full_text = prompt + answer

    tok_prompt = tokenizer(
        prompt,
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding="max_length",
    )
    tok_full = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding="max_length",
    )

    input_ids = tok_full["input_ids"]
    attention_mask = tok_full["attention_mask"]
    prompt_len = sum(tok_prompt["attention_mask"])

    labels = [-100] * MAX_SEQ_LEN
    for i in range(prompt_len, MAX_SEQ_LEN):
        if attention_mask[i] == 1:
            labels[i] = input_ids[i]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


proc_ds = raw_ds.map(
    build_prompt_and_answer,
    remove_columns=raw_ds.column_names,
    desc="Tokenizing and building labels",
)

split = proc_ds.train_test_split(test_size=0.1, seed=42)
train_ds = split["train"]
eval_ds = split["test"]
len(train_ds), len(eval_ds)

## 5. Training Configuration

The recorded v3 run trained for 3 epochs and completed `108` optimizer steps. In the archived run, `eval_steps=200` was larger than the total step count, so validation loss was not emitted. For future reruns, this polished notebook uses epoch-level evaluation so the 90/10 validation split is always measured.

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=32,
    num_train_epochs=3,
    learning_rate=5e-6,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=default_data_collator,
)

## 6. Training Run

The completed recorded run is preserved in `experiments/ft3_recorded_training_run.ipynb`.

**Recorded completed run:**

- Train examples: `2,252`
- Validation examples: `251`
- Epochs: `3`
- Optimizer steps: `108`
- Runtime shown in notebook output: about `23:06`
- Inference sanity check after save: `SELECT AVG(impact_factor) FROM journals;`

In [ ]:
# Rerun only in a GPU environment with enough disk space for checkpoints.
# trainer.train()
# trainer.save_model(OUTPUT_DIR)
# tokenizer.save_pretrained(OUTPUT_DIR)

## 7. Inference Sanity Check

After training, the fine-tuned model is loaded from `OUTPUT_DIR` and prompted with only system/user messages. Generation is greedy for deterministic SQL output.

In [ ]:
def generate_from_ft(example, ft_model, ft_tokenizer, max_new_tokens=256):
    eval_messages = [message for message in example["messages"] if message["role"] != "assistant"]
    prompt = ft_tokenizer.apply_chat_template(
        eval_messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = ft_tokenizer(prompt, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        output_ids = ft_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=ft_tokenizer.pad_token_id,
            eos_token_id=ft_tokenizer.eos_token_id,
        )

    gen_ids = output_ids[0, inputs["input_ids"].shape[1]:]
    return ft_tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

## 8. Spider-Style Evaluation

The external evaluation used Spider dev examples, Spider database schemas, and the official evaluator. Those external assets are ignored and not committed. The archived evaluation notebook is `results/report_ft_v3_spider_eval.ipynb`.

In [ ]:
# Evaluation sketch from the completed run:
# 1. Load spider_data/spider_data/dev.json and tables.json.
# 2. Build schema-aware prompts for each Spider dev question.
# 3. Generate one SQL query per example with greedy decoding.
# 4. Write predictions to spider_official_eval/pred_ft_v3_dev.txt.
# 5. Run official evaluation.py with --etype all.

RECORDED_RESULTS = {
    "base_qwen2_5_coder": {"execution": 0.344, "exact_match": 0.279},
    "fine_tuned_v3": {"execution": 0.359, "exact_match": 0.310},
}
RECORDED_RESULTS

## 9. Results Interpretation

| Model | Spider execution accuracy | Spider exact match |
| --- | ---: | ---: |
| Base Qwen2.5-Coder | 34.4% | 27.9% |
| Fine-tuned v3 | 35.9% | 31.0% |

The fine-tuned model produced a modest improvement on Spider dev: about `+1.5` execution-accuracy points and `+3.1` exact-match points. For a portfolio project, the important signal is the complete fine-tuning workflow: custom dataset formatting, prompt-masked SFT, deterministic inference, baseline comparison, and benchmark evaluation.